In [1]:
# Install wandb for experiment tracking, transformers for AST, timm for ViT
# skimage is used for spectrogram resizing
# Run this cell first on Kaggle before anything else

import subprocess
subprocess.run(["pip", "install", "wandb", "transformers", "timm", "scikit-image", "-q"])

CompletedProcess(args=['pip', 'install', 'wandb', 'transformers', 'timm', 'scikit-image', '-q'], returncode=0)

In [2]:
# Core scientific computing and audio processing libraries

import os
import gc
import json
import random
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from tqdm import tqdm

warnings.filterwarnings("ignore")

# Audio processing
import librosa
import librosa.display
from skimage.transform import resize as sk_resize

# Deep learning
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.models as models
from torch.optim import AdamW
from torch.optim.lr_scheduler import OneCycleLR

# Transformers and ViT (for AST)
import timm
try:
    from transformers import ASTFeatureExtractor, ASTForAudioClassification
    HF_AVAILABLE = True
except ImportError:
    HF_AVAILABLE = False
    print("HuggingFace transformers not available — will use timm ViT as AST substitute")

# Evaluation
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import f1_score, confusion_matrix, classification_report

# WandB experiment tracking
import wandb

# Reproducibility seed
def set_seed(seed=42):
    """Fix all random seeds for reproducibility"""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

# Select GPU if available
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Running on device: {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

2026-03-27 11:35:33.269676: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1774611333.503477      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1774611333.575070      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1774611334.119139      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774611334.119184      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774611334.119187      55 computation_placer.cc:177] computation placer alr

Running on device: cuda
GPU: Tesla T4
VRAM: 15.6 GB


In [3]:
# All hyperparameters and paths are defined here so you can change them easily

CONFIG = {
    # ---- Dataset paths (Kaggle input directory) ----
    "data_dir":          "/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup",
    "genres_stems_dir":  "/kaggle/input/jan-2026-dl-gen-ai-project/messy-mashup/genres_stems",
    "mashups_dir":       "/kaggle/input/jan-2026-dl-gen-ai-project/messy-mashup/mashups",
    "esc50_audio_dir":   "/kaggle/input/jan-2026-dl-gen-ai-project/messy-mashup/ESC-50-master/audio",
    "esc50_meta_csv":    "/kaggle/input/jan-2026-dl-gen-ai-project/messy-mashup/ESC-50-master/meta/esc50.csv",
    "test_csv":          "/kaggle/input/jan-2026-dl-gen-ai-project/messy-mashup/test.csv",
    "sample_sub_csv":    "/kaggle/input/jan-2026-dl-gen-ai-project/messy-mashup/sample_submission.csv",

    # ---- Audio parameters ----
    "sample_rate":   22050,   # Hz — standard for music analysis
    "duration":      20,      # seconds to load per clip
    "n_mels":        128,     # number of mel filterbank channels
    "n_fft":         2048,    # FFT window size (higher = better frequency resolution)
    "hop_length":    512,     # hop between FFT frames (lower = more time resolution)
    "fmax":          8000,    # max frequency for mel filterbank

    # ---- Spectrogram image size fed into models ----
    "img_height":    128,     # = n_mels
    "img_width":     256,     # time frames (resized to this)

    # ---- Training hyperparameters ----
    "batch_size":        32,
    "num_workers":       4,
    "val_split":         0.15,
    "num_epochs_cnn":    50,
    "num_epochs_resnet": 30,
    "num_epochs_ast":    20,
    "lr_cnn":            1e-3,
    "lr_resnet":         5e-4,
    "lr_ast":            1e-4,
    "weight_decay":      1e-4,
    "label_smoothing":   0.1,
    "grad_clip":         1.0,

    # ---- Data augmentation ----
    "aug_per_song":      3,   # how many mashup-style augmented copies per training song
    "num_tta":           5,   # test-time augmentation rounds for prediction

    # ---- Classes ----
    "num_classes": 10,
    "genres": [
        "blues", "classical", "country", "disco", "hiphop",
        "jazz", "metal", "pop", "reggae", "rock"
    ],

    # ---- WandB ----
    "wandb_project": "23f3004501-t12026",
    "wandb_entity":  "23f3004501-indian-institute-of-technology-madras",   
}

print("Configuration loaded successfully!")
print(f"Genres: {CONFIG['genres']}")
print(f"Spectrogram shape: ({CONFIG['img_height']}, {CONFIG['img_width']})")

Configuration loaded successfully!
Genres: ['blues', 'classical', 'country', 'disco', 'hiphop', 'jazz', 'metal', 'pop', 'reggae', 'rock']
Spectrogram shape: (128, 256)


In [4]:
# Set your WandB API key here. You can also add it to Kaggle Secrets.

# --- Set your WandB API key below ---
WANDB_API_KEY = "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"   # <-- REPLACE WITH YOUR ACTUAL KEY
os.environ["WANDB_API_KEY"] = WANDB_API_KEY

def init_wandb_run(run_name: str, extra_config: dict = None):
    """
    Initialize a fresh WandB run. Each model gets its own run so you
    can compare them side-by-side on the WandB dashboard.
    
    Args:
        run_name:     Name for this run (e.g. 'scratch_cnn', 'resnet50', 'ast')
        extra_config: Any extra keys to add to the run config
    Returns:
        wandb.Run object
    """
    cfg = {k: v for k, v in CONFIG.items() if not isinstance(v, list)}
    if extra_config:
        cfg.update(extra_config)

    run = wandb.init(
        project=CONFIG["wandb_project"],
        entity=CONFIG["wandb_entity"],
        name=run_name,
        config=cfg,
        reinit=True,   # allow multiple runs in the same kernel session
    )
    print(f"WandB run started: {run.name} — {run.url}")
    return run

print("WandB helper ready. Remember to replace WANDB_API_KEY with your real key!")

WandB helper ready. Remember to replace WANDB_API_KEY with your real key!


In [5]:
def load_audio(path: str, sr=CONFIG["sample_rate"], duration=CONFIG["duration"]) -> np.ndarray:
    """
    Load a WAV file to a mono float32 numpy array.
    - Pads with zeros if the clip is shorter than `duration` seconds.
    - Truncates if the clip is longer.
    
    Returns: waveform array of length (sr * duration,)
    """
    target_len = sr * duration
    try:
        y, _ = librosa.load(path, sr=sr, duration=duration, mono=True)
        if len(y) < target_len:
            y = np.pad(y, (0, target_len - len(y)), mode="constant")
        else:
            y = y[:target_len]
        return y.astype(np.float32)
    except Exception as e:
        print(f"  [WARN] Could not load {path}: {e}")
        return np.zeros(target_len, dtype=np.float32)


def mel_spectrogram(y: np.ndarray,
                    sr=CONFIG["sample_rate"],
                    n_mels=CONFIG["n_mels"],
                    n_fft=CONFIG["n_fft"],
                    hop_length=CONFIG["hop_length"],
                    fmax=CONFIG["fmax"]) -> np.ndarray:
    """
    Compute a log-mel spectrogram from waveform y.
    
    1. Mel spectrogram (power spectrum on mel scale)
    2. Convert to dB scale (log compression)
    
    Returns: array of shape (n_mels, time_frames)
    """
    S = librosa.feature.melspectrogram(
        y=y, sr=sr, n_mels=n_mels,
        n_fft=n_fft, hop_length=hop_length, fmax=fmax
    )
    return librosa.power_to_db(S, ref=np.max).astype(np.float32)


def normalize_spec(spec: np.ndarray) -> np.ndarray:
    """Min-max normalize spectrogram values to [0, 1]"""
    lo, hi = spec.min(), spec.max()
    if hi - lo > 1e-6:
        return (spec - lo) / (hi - lo)
    return np.zeros_like(spec)


def resize_spec(spec: np.ndarray, h=CONFIG["img_height"], w=CONFIG["img_width"]) -> np.ndarray:
    """
    Resize spectrogram to a fixed (H, W) shape so all model inputs are identical.
    Uses anti-aliased bicubic interpolation from skimage.
    """
    return sk_resize(spec, (h, w), anti_aliasing=True).astype(np.float32)


def wav_to_spec(path: str) -> np.ndarray:
    """Full pipeline: WAV file → normalized, resized mel spectrogram (H, W)"""
    y = load_audio(path)
    spec = mel_spectrogram(y)
    spec = normalize_spec(spec)
    spec = resize_spec(spec)
    return spec


print("Audio utility functions ready!")

Audio utility functions ready!


In [6]:
# Key challenge: training data = clean stems, test data = noisy mashups.
# We bridge this domain gap by synthesizing mashup-like training samples.

def load_esc50_files(esc50_dir: str) -> list:
    """Collect all WAV files from the ESC-50 noise dataset directory"""
    files = list(Path(esc50_dir).glob("*.wav"))
    print(f"Found {len(files)} ESC-50 noise files")
    return files


def mix_cross_song_stems(genre_dir: str) -> np.ndarray:
    """
    Create a synthetic mashup by combining stems from DIFFERENT songs
    of the same genre — exactly how the test mashups are built.
    
    Each of the 4 stems (drums, vocals, bass, others) is taken from
    a randomly chosen song within the genre folder.
    
    Returns: mixed mono waveform of length (sr * duration,)
    """
    sr, duration = CONFIG["sample_rate"], CONFIG["duration"]
    target_len = sr * duration
    song_dirs = [d for d in Path(genre_dir).iterdir() if d.is_dir()]
    
    if len(song_dirs) < 2:
        song_dirs = song_dirs * 4   # safety fallback for small genres

    stem_names = ["drums.wav", "vocals.wav", "bass.wav", "others.wav"]
    mixed = np.zeros(target_len, dtype=np.float32)

    for stem in stem_names:
        # Pick a different random song for each stem
        song_dir = random.choice(song_dirs)
        stem_path = song_dir / stem
        
        # some datasets use "other.wav" instead of "others.wav"
        if not stem_path.exists():
            alt = song_dir / "other.wav"
            if alt.exists():
                stem_path = alt

        if stem_path.exists():
            y = load_audio(str(stem_path))
            # Random volume scaling simulates instrument balance differences
            scale = random.uniform(0.4, 1.6)
            mixed += y * scale

    # Peak-normalize to avoid clipping
    peak = np.max(np.abs(mixed))
    if peak > 1e-6:
        mixed = mixed / peak * 0.9
    return mixed


def add_esc50_noise(y: np.ndarray, noise_files: list, snr_range=(5, 25)) -> np.ndarray:
    """
    Mix a random ESC-50 clip into the audio at a random SNR level (dB).
    
    SNR 5  dB → heavy noise  (test extreme)
    SNR 25 dB → light noise  (test mild)
    10% chance of no noise added.
    
    Returns: noisy waveform, same shape as y
    """
    if not noise_files or random.random() < 0.10:
        return y

    noise_path = random.choice(noise_files)
    noise = load_audio(str(noise_path))

    snr_db    = random.uniform(*snr_range)
    sig_pwr   = np.mean(y ** 2) + 1e-10
    nse_pwr   = np.mean(noise ** 2) + 1e-10
    snr_lin   = 10 ** (snr_db / 10)
    scale     = np.sqrt(sig_pwr / (snr_lin * nse_pwr))

    noisy = y + noise * scale
    peak  = np.max(np.abs(noisy))
    if peak > 1e-6:
        noisy = noisy / peak * 0.9
    return noisy.astype(np.float32)


def tempo_perturb(y: np.ndarray, rate_range=(0.85, 1.15)) -> np.ndarray:
    """
    Stretch or compress time axis (tempo change) while preserving pitch.
    Simulates the BPM alignment done when mixing stems from different songs.
    """
    rate = random.uniform(*rate_range)
    y_out = librosa.effects.time_stretch(y, rate=rate)
    target_len = CONFIG["sample_rate"] * CONFIG["duration"]
    if len(y_out) < target_len:
        y_out = np.pad(y_out, (0, target_len - len(y_out)))
    return y_out[:target_len].astype(np.float32)


def spec_augment(spec: np.ndarray,
                 freq_param=20, time_param=30, n_masks=2) -> np.ndarray:
    """
    SpecAugment: randomly zero out horizontal (frequency) and vertical
    (time) bands in the spectrogram. Proven to reduce overfitting on
    audio tasks (Park et al. 2019).
    
    Args:
        freq_param: max number of mel bins to mask
        time_param: max number of time frames to mask
        n_masks:    how many masks to apply in each dimension
    
    Returns: augmented copy of spec
    """
    spec = spec.copy()
    H, W = spec.shape

    for _ in range(n_masks):
        # Frequency mask
        f  = random.randint(0, freq_param)
        f0 = random.randint(0, max(0, H - f))
        spec[f0:f0 + f, :] = 0.0

        # Time mask
        t  = random.randint(0, time_param)
        t0 = random.randint(0, max(0, W - t))
        spec[:, t0:t0 + t] = 0.0

    return spec


print("Augmentation functions ready!")

Augmentation functions ready!


In [7]:
# This cell crawls genres_stems/, builds original and augmented spectrograms,
# and returns a flat list of (spec_array, label_int) pairs ready for training.

def build_training_dataset(genres_stems_dir: str, esc50_audio_dir: str,
                           aug_per_song: int = CONFIG["aug_per_song"]):
    """
    For every song in every genre:
      1. Original: mix all 4 stems of the SAME song → 1 spectrogram
      2. Augmented (aug_per_song times):
           - Mix stems from DIFFERENT songs of same genre (cross-song)
           - Apply random tempo perturbation (50% chance)
           - Add ESC-50 noise at random SNR

    This domain adaptation strategy is the most important part of the
    solution — it shifts training distribution toward the test distribution.

    Returns:
        samples       : list of (np.ndarray spec, int label)
        label_encoder : fitted sklearn LabelEncoder
    """
    genres       = CONFIG["genres"]
    le           = LabelEncoder().fit(genres)
    noise_files  = load_esc50_files(esc50_audio_dir)
    samples      = []
    sr, dur      = CONFIG["sample_rate"], CONFIG["duration"]
    H, W         = CONFIG["img_height"], CONFIG["img_width"]

    print("\nBuilding training dataset — this takes ~10-20 min on CPU, ~3-5 min on GPU…")

    for genre in tqdm(genres, desc="Genres"):
        genre_dir   = Path(genres_stems_dir) / genre
        label       = int(le.transform([genre])[0])
        song_dirs   = sorted([d for d in genre_dir.iterdir() if d.is_dir()])

        for song_dir in song_dirs:
            # --- 1. Original sample: sum all stems of THIS song ---
            mixed = np.zeros(sr * dur, dtype=np.float32)
            for stem in ["drums.wav", "vocals.wav", "bass.wav", "others.wav"]:
                sp = song_dir / stem
                if sp.exists():
                    mixed += load_audio(str(sp))
            peak = np.max(np.abs(mixed))
            if peak > 1e-6:
                mixed /= peak / 0.9

            spec = normalize_spec(mel_spectrogram(mixed))
            samples.append((resize_spec(spec, H, W), label))

            # --- 2. Augmented samples ---
            for _ in range(aug_per_song):
                aug = mix_cross_song_stems(str(genre_dir))

                if random.random() > 0.5:              # 50%: tempo change
                    aug = tempo_perturb(aug)

                aug  = add_esc50_noise(aug, noise_files)  # add noise
                spec = normalize_spec(mel_spectrogram(aug))
                samples.append((resize_spec(spec, H, W), label))

    print(f"\nTotal training samples: {len(samples)}")
    dist = pd.Series([s[1] for s in samples]).map(
        {i: g for i, g in enumerate(genres)}).value_counts()
    print("Label distribution:\n", dist.to_string())
    return samples, le


# ---- RUN FEATURE EXTRACTION ----
print("Starting feature extraction…")
all_samples, label_encoder = build_training_dataset(
    CONFIG["genres_stems_dir"],
    CONFIG["esc50_audio_dir"],
    aug_per_song=CONFIG["aug_per_song"],
)
print("Feature extraction complete!")

Starting feature extraction…
Found 0 ESC-50 noise files

Building training dataset — this takes ~10-20 min on CPU, ~3-5 min on GPU…


Genres:   0%|          | 0/10 [00:00<?, ?it/s]


FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/input/jan-2026-dl-gen-ai-project/messy-mashup/genres_stems/blues'

In [ ]:
class TrainDataset(Dataset):
    """
    Wraps pre-computed (spec, label) pairs for the training/validation split.
    Applies SpecAugment on-the-fly during training to further regularize.
    
    The spectrogram is replicated across 3 channels so it is compatible
    with ImageNet-pretrained CNNs / ResNets that expect 3-channel input.
    """

    def __init__(self, samples: list, is_train: bool = True):
        self.samples  = samples
        self.is_train = is_train

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        spec, label = self.samples[idx]

        # Online SpecAugment — only during training
        if self.is_train and random.random() > 0.3:
            spec = spec_augment(spec, freq_param=20, time_param=30, n_masks=2)

        # Shape: (1, H, W) → (3, H, W)  [grey → RGB by repeating]
        tensor = torch.from_numpy(spec).unsqueeze(0).repeat(3, 1, 1)
        return tensor, torch.tensor(label, dtype=torch.long)


class TestDataset(Dataset):
    """
    Loads and pre-computes spectrograms for all test mashup files.
    Stored in memory for fast repeated access during TTA inference.
    """

    def __init__(self, file_paths: list):
        self.specs = []
        print("Pre-computing test spectrograms…")
        for fp in tqdm(file_paths, desc="Test audio"):
            self.specs.append(wav_to_spec(fp))
        print(f"Loaded {len(self.specs)} test spectrograms.")

    def __len__(self):
        return len(self.specs)

    def __getitem__(self, idx):
        tensor = torch.from_numpy(self.specs[idx]).unsqueeze(0).repeat(3, 1, 1)
        return tensor


# ---- Train / Val split (stratified) ----
print("\nSplitting data into train / val…")
train_samp, val_samp = train_test_split(
    all_samples,
    test_size=CONFIG["val_split"],
    stratify=[s[1] for s in all_samples],
    random_state=42,
)
print(f"Train: {len(train_samp)} samples  |  Val: {len(val_samp)} samples")

# ---- Create DataLoaders ----
train_dl = DataLoader(TrainDataset(train_samp, is_train=True),
                      batch_size=CONFIG["batch_size"], shuffle=True,
                      num_workers=CONFIG["num_workers"], pin_memory=True,
                      drop_last=True)

val_dl   = DataLoader(TrainDataset(val_samp, is_train=False),
                      batch_size=CONFIG["batch_size"], shuffle=False,
                      num_workers=CONFIG["num_workers"], pin_memory=True)

print(f"Train batches: {len(train_dl)}  |  Val batches: {len(val_dl)}")

# ---- Load test CSV and build test file paths ----
print("\nReading test.csv…")
test_df = pd.read_csv(CONFIG["test_csv"])
print(f"Test samples: {len(test_df)}")
print(test_df.head())

# Map IDs to mashup file paths
# The test CSV has an 'id' column; mashup files are named song0001.wav etc.
test_paths = []
for _, row in test_df.iterrows():
    fname = f"song{str(row['id']).zfill(4)}.wav"
    test_paths.append(os.path.join(CONFIG["mashups_dir"], fname))

found = sum(1 for p in test_paths if os.path.exists(p))
print(f"Test files found: {found}/{len(test_paths)}")

test_ds = TestDataset(test_paths)
test_dl = DataLoader(test_ds, batch_size=CONFIG["batch_size"], shuffle=False,
                     num_workers=CONFIG["num_workers"], pin_memory=True)

print("\nAll data loaders ready!")

In [ ]:
# A pure CNN designed specifically for mel spectrogram classification.
# No pretrained weights — learns entirely from our augmented data.

class ConvBlock(nn.Module):
    """Conv2d → BatchNorm2d → ReLU (with optional Dropout)"""

    def __init__(self, in_ch, out_ch, k=3, s=1, p=1, drop=0.0):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, k, s, p, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Dropout2d(drop) if drop > 0 else nn.Identity(),
        )

    def forward(self, x):
        return self.block(x)


class ScratchCNN(nn.Module):
    """
    Custom 5-block CNN for music genre classification.

    Architecture overview:
        Block 1: 3  → 32  ch  |  128×256 → 64×128
        Block 2: 32 → 64  ch  |  64×128  → 32×64
        Block 3: 64 → 128 ch  |  32×64   → 16×32
        Block 4: 128→ 256 ch  |  16×32   →  8×16
        Block 5: 256→ 512 ch  |   8×16   →  4×8
        GAP: 4×8 → 1×1
        MLP: 512 → 256 → 128 → 10

    Design decisions:
    - Increasing channel width captures richer frequency/time patterns
    - Global Average Pooling avoids overfitting from large FC layers
    - DropOut + Dropout2d regularize both spatial features and neurons
    - Label smoothing in the loss provides soft targets
    """

    def __init__(self, num_classes=10, dropout=0.5):
        super().__init__()

        self.features = nn.Sequential(
            # Block 1 — low-level timbral features
            ConvBlock(3,   32,  drop=0.10),
            ConvBlock(32,  32,  drop=0.00),
            nn.MaxPool2d(2, 2),

            # Block 2 — rhythm and harmonic patterns
            ConvBlock(32,  64,  drop=0.10),
            ConvBlock(64,  64,  drop=0.00),
            nn.MaxPool2d(2, 2),

            # Block 3 — mid-level structure
            ConvBlock(64,  128, drop=0.15),
            ConvBlock(128, 128, drop=0.00),
            nn.MaxPool2d(2, 2),

            # Block 4 — genre-discriminative features
            ConvBlock(128, 256, drop=0.20),
            ConvBlock(256, 256, drop=0.00),
            nn.MaxPool2d(2, 2),

            # Block 5 — high-level abstract features
            ConvBlock(256, 512, drop=0.25),
            ConvBlock(512, 512, drop=0.00),
            nn.MaxPool2d(2, 2),
        )

        # Global Average Pooling collapses spatial dims → (B, 512)
        self.gap = nn.AdaptiveAvgPool2d(1)

        # MLP classifier
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(512, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(256, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout * 0.5),
            nn.Linear(128, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.gap(x)
        x = self.classifier(x)
        return x


# Quick sanity check for CNN
_cnn_test = ScratchCNN(num_classes=10).to(DEVICE)
_dummy    = torch.randn(2, 3, 128, 256).to(DEVICE)
print("CNN output shape:", _cnn_test(_dummy).shape)
print(f"CNN parameters: {sum(p.numel() for p in _cnn_test.parameters()):,}")
del _cnn_test, _dummy
torch.cuda.empty_cache()

In [ ]:
# Pretrained on ImageNet — repurposed for mel spectrogram classification.
# We replicate the single-channel spectrogram to 3 channels so ResNet's
# first Conv layer receives the expected 3-channel input.

class ResNet50Genre(nn.Module):
    """
    ResNet-50 fine-tuned for 10-class music genre classification.

    Transfer learning strategy (two-stage):
      Stage 1 (epochs 1-10):  Freeze layer1, layer2, stem.  Only train
                               layer3, layer4, and the new FC head.
      Stage 2 (epoch 11+):    Unfreeze ALL layers.  Reduce LR 10×.

    The custom FC head adds an extra bottleneck:
        2048 → 512 → 256 → 10
    with Dropout for regularization.
    """

    def __init__(self, num_classes=10, pretrained=True, dropout=0.5):
        super().__init__()

        # Load backbone
        self.backbone = models.resnet50(pretrained=pretrained)
        feat_dim = self.backbone.fc.in_features  # 2048

        # Replace final fully-connected layer with a deeper head
        self.backbone.fc = nn.Sequential(
            nn.Linear(feat_dim, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(512, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout * 0.5),
            nn.Linear(256, num_classes),
        )

        # Freeze early layers during Stage 1 (fine-tune only deep layers)
        self._freeze_early()

    def _freeze_early(self):
        """Freeze stem + layer1 + layer2 (low-level ImageNet features)"""
        for name, param in self.backbone.named_parameters():
            if any(x in name for x in ["conv1", "bn1", "layer1", "layer2"]):
                param.requires_grad = False

    def unfreeze_all(self):
        """Unfreeze every parameter for full end-to-end fine-tuning"""
        for p in self.backbone.parameters():
            p.requires_grad = True

    def forward(self, x):
        return self.backbone(x)


# Sanity check for ResNet-50
_rn_test  = ResNet50Genre(num_classes=10).to(DEVICE)
_dummy    = torch.randn(2, 3, 128, 256).to(DEVICE)
print("ResNet-50 output shape:", _rn_test(_dummy).shape)
trainable = sum(p.numel() for p in _rn_test.parameters() if p.requires_grad)
total     = sum(p.numel() for p in _rn_test.parameters())
print(f"ResNet-50 trainable params: {trainable:,} / {total:,} total")
del _rn_test, _dummy
torch.cuda.empty_cache()

In [ ]:
# AST treats the 2-D mel spectrogram as a sequence of non-overlapping
# patches and processes them with multi-head self-attention — the same
# idea as ViT for images but pre-trained on audio.

class ASTGenreModel(nn.Module):
    """
    Audio Spectrogram Transformer for music genre classification.

    Primary:  MIT/ast-finetuned-audioset-10-10-0.4593 (HuggingFace)
    Fallback: vit_small_patch16_224 from timm if HF is unavailable

    The HF model was pretrained on AudioSet (2M audio clips, 527 classes).
    We replace the classification head and fine-tune, freezing the
    first 4 of 12 transformer encoder layers to prevent catastrophic
    forgetting on the small dataset.

    Input: (B, 3, 128, 256) — we use only the first channel (grey).
    Output: (B, num_classes) logits
    """

    def __init__(self, num_classes=10,
                 hf_model="MIT/ast-finetuned-audioset-10-10-0.4593"):
        super().__init__()
        self.use_hf = False

        if HF_AVAILABLE:
            try:
                # Attempt to load pretrained AST from HuggingFace hub
                self.model = ASTForAudioClassification.from_pretrained(
                    hf_model,
                    num_labels=num_classes,
                    ignore_mismatched_sizes=True,
                )
                # Freeze first 4 transformer layers
                for i, layer in enumerate(
                    self.model.audio_spectrogram_transformer.encoder.layer
                ):
                    if i < 4:
                        for p in layer.parameters():
                            p.requires_grad = False

                self.use_hf = True
                print("Loaded HuggingFace AST model ✓")
            except Exception as e:
                print(f"HF AST load failed ({e}) — falling back to ViT-Small")

        if not self.use_hf:
            # Fallback: ViT-Small (similar transformer architecture for spectrograms)
            self.model = timm.create_model(
                "vit_small_patch16_224",
                pretrained=True,
                num_classes=num_classes,
                img_size=(128, 256),   # custom resolution for our spectrograms
            )
            # Freeze first 4 transformer blocks
            for blk in self.model.blocks[:4]:
                for p in blk.parameters():
                    p.requires_grad = False
            print("Using timm ViT-Small as AST substitute ✓")

    def forward(self, x):
        if self.use_hf:
            # HF AST expects single-channel input: (B, 1, freq, time)
            x_in = x[:, 0:1, :, :]
            return self.model(input_values=x_in).logits
        else:
            # timm ViT expects 3-channel input: (B, 3, H, W)
            return self.model(x)


# Load and test AST / ViT model
print("Initializing AST model…")
_ast_test = ASTGenreModel(num_classes=10).to(DEVICE)
_dummy    = torch.randn(2, 3, 128, 256).to(DEVICE)
print("AST output shape:", _ast_test(_dummy).shape)
trainable = sum(p.numel() for p in _ast_test.parameters() if p.requires_grad)
total     = sum(p.numel() for p in _ast_test.parameters())
print(f"AST trainable: {trainable:,} / {total:,}")
del _ast_test, _dummy
torch.cuda.empty_cache()

In [ ]:
def train_epoch(model, loader, optimizer, criterion, scaler, device):
    """
    One full pass over the training set.
    Uses mixed-precision (AMP) for faster GPU computation.
    Clips gradients to prevent exploding-gradient instability.
    
    Returns: (avg_loss, accuracy, macro_f1)
    """
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    all_preds, all_labels = [], []

    for specs, labels in tqdm(loader, desc="  Train", leave=False):
        specs, labels = specs.to(device), labels.to(device)
        optimizer.zero_grad(set_to_none=True)

        # Forward + loss with AMP
        with torch.cuda.amp.autocast():
            logits = model(specs)
            loss   = criterion(logits, labels)

        # Backward + gradient clip + optimizer step
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), CONFIG["grad_clip"])
        scaler.step(optimizer)
        scaler.update()

        # Metrics bookkeeping
        total_loss += loss.item()
        preds = logits.argmax(dim=1)
        correct += preds.eq(labels).sum().item()
        total   += labels.size(0)
        all_preds .extend(preds .cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

    avg_loss = total_loss / len(loader)
    acc      = correct / total
    f1       = f1_score(all_labels, all_preds, average="macro", zero_division=0)
    return avg_loss, acc, f1


@torch.no_grad()
def eval_epoch(model, loader, criterion, device):
    """
    One full pass over the validation set.
    No gradient computation — faster and uses less memory.
    
    Returns: (avg_loss, accuracy, macro_f1)
    """
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    all_preds, all_labels = [], []

    for specs, labels in tqdm(loader, desc="  Val  ", leave=False):
        specs, labels = specs.to(device), labels.to(device)

        with torch.cuda.amp.autocast():
            logits = model(specs)
            loss   = criterion(logits, labels)

        total_loss += loss.item()
        preds = logits.argmax(dim=1)
        correct += preds.eq(labels).sum().item()
        total   += labels.size(0)
        all_preds .extend(preds .cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

    avg_loss = total_loss / len(loader)
    acc      = correct / total
    f1       = f1_score(all_labels, all_preds, average="macro", zero_division=0)
    return avg_loss, acc, f1


def run_training(model, model_key: str, num_epochs: int, lr: float,
                 wd=CONFIG["weight_decay"]):
    """
    Full training loop for any of the three models.
    
    Features:
      - AdamW optimizer (decoupled weight decay)
      - OneCycleLR scheduler (warm-up + cosine annealing)
      - Mixed-precision AMP training
      - CrossEntropy loss with label smoothing (0.1)
      - WandB metric logging (per batch + per epoch)
      - Best-model checkpointing (saved by val macro F1)
      - Stage 2 unfreeze for ResNet-50 at epoch 11
    
    Returns:
        model      : loaded with best-epoch weights
        history    : dict of lists (train/val loss, acc, f1)
        best_val_f1: float
    """
    print(f"\n{'='*58}")
    print(f"  Training: {model_key.upper()}")
    print(f"  Epochs: {num_epochs}  |  LR: {lr}  |  WD: {wd}")
    print(f"{'='*58}")

    # WandB run for this model
    wb = init_wandb_run(model_key, {"lr": lr, "wd": wd, "epochs": num_epochs})

    criterion  = nn.CrossEntropyLoss(label_smoothing=CONFIG["label_smoothing"])
    optimizer  = AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=lr, weight_decay=wd
    )
    scheduler  = OneCycleLR(
        optimizer, max_lr=lr,
        epochs=num_epochs, steps_per_epoch=len(train_dl),
        pct_start=0.1, anneal_strategy="cos",
    )
    scaler     = torch.cuda.amp.GradScaler()

    best_f1    = 0.0
    ckpt_path  = f"/kaggle/working/best_{model_key}.pth"
    history    = {k: [] for k in
                  ["train_loss","val_loss","train_acc","val_acc","train_f1","val_f1"]}

    for epoch in range(1, num_epochs + 1):

        # ResNet-50 Stage 2: full fine-tune from epoch 11 onward
        if model_key == "resnet50" and epoch == 11:
            print("  [Stage 2] Unfreezing all ResNet-50 layers…")
            model.unfreeze_all()
            for pg in optimizer.param_groups:
                pg["lr"] = lr * 0.1   # lower LR for full fine-tune

        # ---- Train ----
        tr_loss, tr_acc, tr_f1 = train_epoch(
            model, train_dl, optimizer, criterion, scaler, DEVICE)
        scheduler.step()

        # ---- Validate ----
        vl_loss, vl_acc, vl_f1 = eval_epoch(
            model, val_dl, criterion, DEVICE)

        # ---- WandB logging ----
        wb.log({
            "epoch":            epoch,
            "train/loss":       tr_loss,
            "train/accuracy":   tr_acc,
            "train/macro_f1":   tr_f1,
            "val/loss":         vl_loss,
            "val/accuracy":     vl_acc,
            "val/macro_f1":     vl_f1,
            "lr":               optimizer.param_groups[0]["lr"],
        })

        # ---- Save history ----
        for k, v in zip(
            ["train_loss","val_loss","train_acc","val_acc","train_f1","val_f1"],
            [tr_loss, vl_loss, tr_acc, vl_acc, tr_f1, vl_f1]
        ):
            history[k].append(v)

        # ---- Checkpoint best model ----
        if vl_f1 > best_f1:
            best_f1 = vl_f1
            torch.save(model.state_dict(), ckpt_path)
            wb.summary["best_val_macro_f1"] = best_f1
            marker = " ← BEST"
        else:
            marker = ""

        print(
            f"  Ep {epoch:02d}/{num_epochs} | "
            f"TrLoss {tr_loss:.4f}  TrF1 {tr_f1:.4f} | "
            f"VlLoss {vl_loss:.4f}  VlF1 {vl_f1:.4f}{marker}"
        )

    # Load best weights before returning
    model.load_state_dict(torch.load(ckpt_path, map_location=DEVICE))
    print(f"\n  Best Val Macro F1 ({model_key}): {best_f1:.4f}")

    # Save model artifact to WandB
    artifact = wandb.Artifact(f"model_{model_key}", type="model")
    artifact.add_file(ckpt_path)
    wb.log_artifact(artifact)
    wb.finish()

    return model, history, best_f1


print("Training engine ready!")

In [ ]:
print("\n>>> TRAINING MODEL 1: Custom CNN (from scratch) <<<")

cnn_model = ScratchCNN(num_classes=CONFIG["num_classes"], dropout=0.5).to(DEVICE)

cnn_model, cnn_hist, cnn_best = run_training(
    model=cnn_model,
    model_key="scratch_cnn",
    num_epochs=CONFIG["num_epochs_cnn"],
    lr=CONFIG["lr_cnn"],
)

print(f"\n[CNN] Best Val Macro F1 = {cnn_best:.4f}")

In [ ]:
print("\n>>> TRAINING MODEL 2: ResNet-50 (Transfer Learning) <<<")

resnet_model = ResNet50Genre(
    num_classes=CONFIG["num_classes"],
    pretrained=True,
    dropout=0.5
).to(DEVICE)

resnet_model, resnet_hist, resnet_best = run_training(
    model=resnet_model,
    model_key="resnet50",
    num_epochs=CONFIG["num_epochs_resnet"],
    lr=CONFIG["lr_resnet"],
)

print(f"\n[ResNet-50] Best Val Macro F1 = {resnet_best:.4f}")

In [ ]:
print("\n>>> TRAINING MODEL 3: Audio Spectrogram Transformer (AST) <<<")

ast_model = ASTGenreModel(num_classes=CONFIG["num_classes"]).to(DEVICE)

ast_model, ast_hist, ast_best = run_training(
    model=ast_model,
    model_key="ast",
    num_epochs=CONFIG["num_epochs_ast"],
    lr=CONFIG["lr_ast"],
    wd=1e-2,   # higher weight decay helps transformer fine-tuning
)

print(f"\n[AST] Best Val Macro F1 = {ast_best:.4f}")

In [ ]:
# For every test sample we run `num_tta` forward passes.
# Pass 1: clean spectrogram.
# Pass 2-N: SpecAugmented versions (light masking).
# We average the softmax probabilities across all passes.

@torch.no_grad()
def predict_with_tta(model, loader, device, n_tta=CONFIG["num_tta"]) -> np.ndarray:
    """
    Generate soft (probability) predictions with Test-Time Augmentation.
    
    Args:
        model:  trained model in eval mode
        loader: DataLoader over the test set
        n_tta:  number of augmentation rounds (1 = no TTA)
    
    Returns:
        probs: np.ndarray of shape (N_test, num_classes) — averaged probs
    """
    model.eval()
    all_pass_probs = []

    for tta_i in range(n_tta):
        round_probs = []

        for batch in tqdm(loader, desc=f"TTA {tta_i+1}/{n_tta}", leave=False):
            # TestDataset returns only specs (no label)
            if isinstance(batch, (list, tuple)):
                specs = batch[0].to(device)
            else:
                specs = batch.to(device)

            # Apply light SpecAugment for rounds 2+
            if tta_i > 0:
                np_specs = specs.cpu().numpy()
                aug_batch = []
                for s in np_specs:
                    # Augment the first channel (grey-scale) then re-stack
                    s_aug = spec_augment(
                        s[0], freq_param=10, time_param=20, n_masks=1
                    )
                    aug_batch.append(np.stack([s_aug, s_aug, s_aug], axis=0))
                specs = torch.FloatTensor(np.array(aug_batch)).to(device)

            with torch.cuda.amp.autocast():
                logits = model(specs)
            probs = F.softmax(logits, dim=1).cpu().numpy()
            round_probs.append(probs)

        all_pass_probs.append(np.vstack(round_probs))

    return np.mean(all_pass_probs, axis=0)   # average across TTA rounds


print("Generating predictions for all 3 models with TTA…")

print("\n[1/3] CNN…")
cnn_probs    = predict_with_tta(cnn_model,    test_dl, DEVICE)

print("\n[2/3] ResNet-50…")
resnet_probs = predict_with_tta(resnet_model, test_dl, DEVICE)

print("\n[3/3] AST…")
ast_probs    = predict_with_tta(ast_model,    test_dl, DEVICE)

print("\nAll predictions generated!")

In [ ]:
# Weight each model's probabilities proportional to its val Macro F1 score.
# Better models naturally get more influence in the ensemble.

total_f1  = cnn_best + resnet_best + ast_best
w_cnn     = cnn_best    / total_f1
w_resnet  = resnet_best / total_f1
w_ast     = ast_best    / total_f1

print(f"\nEnsemble weights (proportional to Val Macro F1):")
print(f"  Custom CNN : {w_cnn:.4f}")
print(f"  ResNet-50  : {w_resnet:.4f}")
print(f"  AST / ViT  : {w_ast:.4f}")

# Weighted average of probability vectors
ensemble_probs = w_cnn * cnn_probs + w_resnet * resnet_probs + w_ast * ast_probs

# Argmax → integer label → string genre name
pred_indices = ensemble_probs.argmax(axis=1)
pred_genres  = label_encoder.inverse_transform(pred_indices)

# Build submission DataFrame
submission = pd.DataFrame({
    "id":    test_df["id"].values,
    "genre": pred_genres,
})

print(f"\nSubmission shape: {submission.shape}")
print(submission.head(10))
print("\nPredicted genre distribution:")
print(submission["genre"].value_counts().to_string())

# Save submission CSV
sub_path = "/kaggle/working/submission.csv"
submission.to_csv(sub_path, index=False)
print(f"\nSubmission saved → {sub_path}")

In [ ]:
def plot_history(history: dict, title: str, save_path: str = None):
    """
    Plot loss and macro-F1 curves for train and validation.
    Saves the figure and optionally displays it.
    """
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle(title, fontsize=14, fontweight="bold")

    epochs = range(1, len(history["train_loss"]) + 1)

    # Loss curves
    axes[0].plot(epochs, history["train_loss"], "b-", label="Train")
    axes[0].plot(epochs, history["val_loss"],   "r-", label="Val")
    axes[0].set_xlabel("Epoch");  axes[0].set_ylabel("Cross-Entropy Loss")
    axes[0].set_title("Loss");    axes[0].legend();  axes[0].grid(alpha=0.3)

    # Macro F1 curves
    axes[1].plot(epochs, history["train_f1"], "b-", label="Train")
    axes[1].plot(epochs, history["val_f1"],   "r-", label="Val")
    axes[1].set_xlabel("Epoch");  axes[1].set_ylabel("Macro F1 Score")
    axes[1].set_title("Macro F1"); axes[1].legend(); axes[1].grid(alpha=0.3)

    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()
    return fig


# Plot all three models
plot_history(cnn_hist,    "Custom CNN — Training History",  "/kaggle/working/cnn_history.png")
plot_history(resnet_hist, "ResNet-50 — Training History",   "/kaggle/working/resnet_history.png")
plot_history(ast_hist,    "AST/ViT — Training History",     "/kaggle/working/ast_history.png")


def plot_confusion_matrix(model, loader, device, le, model_name: str, save_path=None):
    """
    Generate and display the confusion matrix on the validation set.
    Helps identify which genre pairs the model confuses most.
    """
    model.eval()
    all_preds, all_labels = [], []

    with torch.no_grad():
        for specs, labels in loader:
            specs = specs.to(device)
            with torch.cuda.amp.autocast():
                logits = model(specs)
            all_preds .extend(logits.argmax(1).cpu().numpy())
            all_labels.extend(labels.numpy())

    cm = confusion_matrix(all_labels, all_preds)
    fig, ax = plt.subplots(figsize=(11, 9))
    sns.heatmap(
        cm, annot=True, fmt="d", cmap="Blues",
        xticklabels=le.classes_,
        yticklabels=le.classes_,
        ax=ax
    )
    ax.set_xlabel("Predicted");  ax.set_ylabel("True")
    ax.set_title(f"Confusion Matrix — {model_name}")
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()

    # Print classification report
    print(f"\nClassification Report — {model_name}:")
    print(classification_report(all_labels, all_preds,
                                target_names=le.classes_, zero_division=0))


# Confusion matrices for all three models
plot_confusion_matrix(cnn_model,    val_dl, DEVICE, label_encoder,
                      "Custom CNN",  "/kaggle/working/cm_cnn.png")
plot_confusion_matrix(resnet_model, val_dl, DEVICE, label_encoder,
                      "ResNet-50",   "/kaggle/working/cm_resnet.png")
plot_confusion_matrix(ast_model,    val_dl, DEVICE, label_encoder,
                      "AST/ViT",     "/kaggle/working/cm_ast.png")

In [ ]:
# Log ensemble results and upload submission to WandB for full traceability.

print("\nLogging final ensemble summary to WandB…")

summary_run = wandb.init(
    project=CONFIG["wandb_project"],
    name="ensemble_summary",
    config={
        "cnn_best_f1":       cnn_best,
        "resnet_best_f1":    resnet_best,
        "ast_best_f1":       ast_best,
        "w_cnn":             float(w_cnn),
        "w_resnet":          float(w_resnet),
        "w_ast":             float(w_ast),
        "num_test_samples":  len(submission),
    },
    reinit=True,
)

# Log final metrics table
summary_run.log({
    "cnn/val_macro_f1":    cnn_best,
    "resnet/val_macro_f1": resnet_best,
    "ast/val_macro_f1":    ast_best,
})

# Upload submission as a WandB artifact
sub_artifact = wandb.Artifact("submission", type="submission")
sub_artifact.add_file(sub_path)
summary_run.log_artifact(sub_artifact)

# Upload training plots as WandB images
summary_run.log({
    "plot/cnn_history":     wandb.Image("/kaggle/working/cnn_history.png"),
    "plot/resnet_history":  wandb.Image("/kaggle/working/resnet_history.png"),
    "plot/ast_history":     wandb.Image("/kaggle/working/ast_history.png"),
    "plot/cm_cnn":          wandb.Image("/kaggle/working/cm_cnn.png"),
    "plot/cm_resnet":       wandb.Image("/kaggle/working/cm_resnet.png"),
    "plot/cm_ast":          wandb.Image("/kaggle/working/cm_ast.png"),
})

summary_run.finish()

In [ ]:
print("\n" + "=" * 58)
print("  FINAL RESULTS SUMMARY")
print("=" * 58)
print(f"  {'Model':<22} {'Val Macro F1':>15}  {'Ensemble Weight':>16}")
print(f"  {'-'*54}")
print(f"  {'Custom CNN (Scratch)':<22} {cnn_best:>15.4f}  {w_cnn:>16.4f}")
print(f"  {'ResNet-50':<22} {resnet_best:>15.4f}  {w_resnet:>16.4f}")
print(f"  {'AST / ViT':<22} {ast_best:>15.4f}  {w_ast:>16.4f}")
print(f"  {'─'*54}")
print(f"  Submission file  : {sub_path}")
print(f"  Predicted samples: {len(submission)}")
print("=" * 58)
print("\nAll done!  Upload submission.csv to Kaggle to get your leaderboard score.")